# Finetune VisDrone — **một model, một tab**

Notebook này train **đúng một model**. Mở 4 tab Colab, mỗi tab chạy notebook
này, **chỉ đổi một dòng duy nhất** ở cell 1:

```python
MODEL = "v8n-base"     # tab 1
MODEL = "v11n-base"    # tab 2
MODEL = "v26n-base"    # tab 3
MODEL = "v26n-p2"      # tab 4
```

Toàn bộ phần còn lại giữ nguyên, không sửa gì. Mỗi tab một GPU, mỗi tab lưu
vào thư mục Drive riêng nên không đè lên nhau. Cell cuối gom kết quả từ mọi
model đã xong thành một bảng.

> **Colab giới hạn số session chạy cùng lúc** (Pro thường 2–3). Mở 4 tab mà bị
> từ chối thì chạy 2 tab trước, xong rồi chạy 2 tab sau — kết quả gom vẫn đủ.

---

## Bốn điều đã kiểm chứng trên `ultralytics==8.4.118`

Không phải suy đoán; mỗi dòng dưới đây đều đo được, và mỗi dòng đều đổi một
chỗ trong notebook.

### 1. `yolo26-p2.yaml` có sẵn trong package — không cần tải

File trong `ultralytics==8.4.118` khớp đúng bản trên GitHub main:
`end2end: True`, `reg_max: 1`, `Detect(P2, P3, P4, P5)` tại `[19, 22, 25, 28]`,
scale `n` → **2.662.400 tham số** (dựng thử ra đúng 2.66M).

### 2. YOLO26 **không dùng NMS** — output khác hẳn

| Model | Output thô | NMS |
|---|---|---|
| `yolov8n`, `yolo11n` | `(1, 4+nc, 8400)` | cần |
| **`yolo26n`, `yolo26n-p2`** | **`(1, 300, 6)`** | **không** |

`(1, 300, 6)` là box decode sẵn `[x1, y1, x2, y2, conf, cls]`.

**Lợi:** pipeline hiện tại tốn **18 ms/frame** NMS trên CPU, so với 47 ms
inference — bỏ được là khoản cắt lớn nhất còn lại.
**Phải xử lý:** cắm v26 vào `3-pipeline/detector.py` sẽ **sai thầm lặng**,
không lỗi. Cell export ghi rõ shape từng model.

### 3. Đầu end-to-end cắt cứng ở **300** detection

Giao thức VisDrone chấm ở `maxDets=500`, mà val của bạn trung bình **345
det/ảnh** ở conf 0.001. Để mặc định thì v26 bị thiệt mà không có dấu hiệu gì.
Notebook nâng `max_det = 500`, đã verify output đổi thành `(1, 500, 6)`.

### 4. `yolo26n-p2.pt` **không tồn tại** — chỉ 40% trọng số nạp được

Phải dựng từ yaml rồi nạp một phần từ `yolo26n.pt`. Đo thật, bằng cách tải
pretrained về rồi đếm tensor trùng tên **và** trùng shape:

| Model | Trọng số nạp được |
|---|---|
| `v8n-base` | 355/355 — **100%** |
| `v11n-base` | 499/499 — **100%** |
| `v26n-base` | 708/708 — **100%** |
| **`v26n-p2`** | **360/902 — 40%** |

Tức **60% model p2 khởi tạo ngẫu nhiên**, trong khi ba model kia nạp đủ. Nên
p2 mặc định được **1.5× epoch**, và tỉ lệ này được ghi vào `summary.json` rồi
lên bảng so sánh. Thiếu con số đó, bảng sẽ bị đọc thành "kiến trúc p2 kém hơn",
trong khi thực ra nó chỉ xuất phát sau.

Thêm: p2 có stride `[4, 8, 16, 32]` thay vì `[8, 16, 32]` → **34.000 anchor**
thay vì 8.400 ở 640px (đo được). Đó là lý do batch mặc định của p2 thấp hơn —
và cũng là lý do nó đáng thử với VisDrone, nơi vật thể rất nhỏ.

## 1. Chọn model — **dòng duy nhất cần sửa giữa các tab**

In [ ]:
# ============================================================
#  DOI DUNG DONG NAY O MOI TAB. Khong sua gi khac.
# ============================================================
MODEL = "v8n-base"        # "v8n-base" | "v11n-base" | "v26n-base" | "v26n-p2"
# ============================================================

SEED = 0
IMGSZ = 640
EPOCHS_BASE = 50

# Batch dat cung cho CA BON tab. Day khong phai chuyen toc do ma la chuyen
# so sanh duoc: so buoc cap nhat gradient = so_anh / batch * epochs. Hai model
# khac batch la khac so buoc train, va luc do bang mAP do "con nao duoc train
# nhieu hon" chu khong phai "kien truc nao tot hon".
#
# Dat None thi cell 9 tu do batch lon nhat vua VRAM -- nhanh hon, nhung moi
# tab ra mot batch khac nhau va bang so sanh mat y nghia.
BATCH_OVERRIDE = 32

# Early stopping: dung khi mAP50-95 khong cai thien sau PATIENCE epoch lien
# tiep. Ultralytics tu lam, chi can truyen patience -- va no luu lai best.pt
# cua epoch tot nhat chu khong phai epoch cuoi, nen dung som khong mat gi.
#
# 15 chu khong phai 30: patience phai nho hon nhieu so voi tong so epoch, neu
# khong thi no khong bao gio kip kich hoat. Voi 50 epoch thi patience 30 nghia
# la phai te lien tuc tu epoch 20 tro di moi dung -- gan nhu khong xay ra, tuc
# la co early stopping tren giay ma thuc te khong bao gio chay.
PATIENCE = 15

# maxDets cua giao thuc VisDrone. Dau end2end mac dinh cat o 300, ma anh
# VisDrone dong co the vuot 300 vat the -> khong nang len la v26 bi thiet
# ma khong bao gi.
MAX_DET = 500

# ---- Cat anh thanh o chong lan ----------------------------------------
# Frame 1920x1080 letterbox ve 640 la co 0.33x: nguoi di bo 15px con 5px, roi
# mosaic co them 2x nua con 2.5px -- duoi nguong ma stride-8 bieu dien duoc.
# Cat 2x2 chong lan 20% cho moi o 1067x600, gain 0.60 thay vi 0.33, tuc vat
# the to len dung 1.80x. Doi lai: 4x so anh, 4x thoi gian epoch, va inference
# phai chay 4 lan roi gop.
TILE = True
TILE_GRID = 2            # 2x2 = 4 o
TILE_OVERLAP = 0.20      # 20% chong lan giua hai o ke nhau

# Box vat qua canh o bi cat lai; giu neu con >= ti le nay. Bo nguong nay thi
# mot mep xe cung thanh nhan "xe", va model se bao xe o moi manh kim loai.
MIN_BOX_VISIBLE = 0.40

# Chi xao tron THU TU trong tap train. KHONG dung den viec chia train/val:
# frame lien nhau trong video gan nhu trung khop, tron roi chia lai la de model
# thay truoc anh val -> mAP dep gia tao.
SHUFFLE_SEED = 0

# ---- Dong bang bao nhieu tang -----------------------------------------
#   0  = train toan bo (3.157M tham so)
#   10 = dong backbone, train neck + head (1.885M, 59.7%)
#   22 = dong backbone + neck, chi train 3 nhanh Detect (0.898M, 28.4%)
#
# Ultralytics dong bang that: requires_grad=False VA .eval() cho BatchNorm
# trong phan dong bang, nen thong ke BN cung khong troi.
FREEZE = 0

REGISTRY = {
    "v8n-base":  dict(cfg="yolov8n.yaml",    weights="yolov8n.pt",
                      batch=128, epochs=EPOCHS_BASE),
    "v11n-base": dict(cfg="yolo11n.yaml",    weights="yolo11n.pt",
                      batch=128, epochs=EPOCHS_BASE),
    "v26n-base": dict(cfg="yolo26n.yaml",    weights="yolo26n.pt",
                      batch=128, epochs=EPOCHS_BASE),
    # Khong co yolo26n-p2.pt -> dung tu yaml, nap mot phan tu yolo26n.pt.
    # Do that: chi 360/902 tensor nap duoc (40%), tuc 60% model khoi tao ngau
    # nhien, trong khi 3 model kia nap du 100%. -> can nhieu epoch hon.
    # Them tang P2 (stride 4) -> 34000 anchor thay vi 8400 -> batch thap hon.
    "v26n-p2":   dict(cfg="yolo26n-p2.yaml", weights="yolo26n.pt",
                      batch=64,  epochs=int(EPOCHS_BASE * 1.5)),
}

assert MODEL in REGISTRY, f"MODEL phai la mot trong {list(REGISTRY)}"
SPEC = dict(REGISTRY[MODEL])

print(f"Tab nay train : {MODEL}")
print(f"  cfg      : {SPEC['cfg']}")
print(f"  weights  : {SPEC['weights']}")
print(f"  batch    : {SPEC['batch']} (cell 9 se do lai va tu chinh)")
print(f"  epochs   : {SPEC['epochs']} (toi da)")
print(f"  patience : {PATIENCE} -> dung som neu {PATIENCE} epoch lien tiep khong cai thien")
print(f"  max_det  : {MAX_DET}")
print(f"  freeze   : {FREEZE}  " + {0: "(train toan bo)", 10: "(dong backbone)",
                                    22: "(chi train Detect head)"}.get(FREEZE, ""))

## 2. Kiểm tra GPU

In [ ]:
import subprocess

print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
     "--format=csv,noheader"],
    capture_output=True, text=True).stdout or "!! khong thay nvidia-smi")

import torch
if torch.cuda.device_count() == 0:
    raise SystemExit("Khong co GPU. Runtime > Change runtime type > GPU (A100).")

GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\nGPU  : {GPU_NAME}")
print(f"VRAM : {VRAM_GB:.1f} GB")

if "A100" not in GPU_NAME:
    print(f"\n[canh bao] Khong phai A100. Batch mac dinh tinh cho A100 40GB; "
          f"cell 9 se do lai va ha xuong cho vua {VRAM_GB:.0f} GB.")

## 3. Cài đặt

Ghim đúng version đã kiểm chứng. `yolo26` chỉ có từ ultralytics 8.4.x — bản cũ
hơn báo "model not found" cho hai model v26.

In [ ]:
%pip install -q "ultralytics==8.4.118" onnx onnxslim onnxruntime

import ultralytics, torch, platform
print("ultralytics", ultralytics.__version__)
print("torch      ", torch.__version__, "| cuda", torch.version.cuda)
print("python     ", platform.python_version())

from ultralytics.utils.downloads import GITHUB_ASSETS_NAMES
w = SPEC["weights"]
print(f"\n{w}: {'co pretrained' if w in GITHUB_ASSETS_NAMES else 'KHONG co'}")
if MODEL == "v26n-p2":
    print("  (dung: yolo26n-p2.pt khong ton tai. Dung yolo26n.pt nap mot phan.)")

## 4. Mount Drive

**Trước khi chạy**: mở link dataset → **Add shortcut to Drive → My Drive**.
Thư mục chia sẻ không tự xuất hiện trong `MyDrive` nếu chưa tạo shortcut, và
`gdown --folder` bị chặn ở 50 file nên vô dụng với dataset vài nghìn ảnh.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
ROOT = "/content/drive/MyDrive"
print("Thu muc cap 1 trong MyDrive:\n")
for d in sorted(os.listdir(ROOT))[:60]:
    if os.path.isdir(os.path.join(ROOT, d)):
        print("  ", d)

## 5. Lấy dataset — chọn **một trong hai cách**

Dataset nằm ở **"Được chia sẻ với tôi"**, và Colab không mount được mục đó.
Không phải vì Colab thiếu tính năng: **"Được chia sẻ với tôi" không phải một
thư mục**, nó là *bộ lọc hiển thị* — không có đường dẫn thật để mount. Colab
chỉ mount `MyDrive` và `Shared drives`.

### Cách A — tải thẳng bằng link (khuyến nghị, **không đụng Drive của bạn**)

Dán link chia sẻ của 2 file zip vào `ZIP_LINKS`. Tải thẳng từ máy chủ Google
về Colab, **không qua FUSE nên nhanh hơn hẳn** cách B, và không thêm gì vào
Drive của bạn.

Lấy link: chuột phải từng file `.zip` → **Chia sẻ** → **Sao chép đường liên
kết**. Cần quyền là **"Bất kỳ ai có đường liên kết"** — nếu đang để "Bị hạn
chế" thì nhờ `votinh42069` đổi, hoặc dùng cách B.

### Cách B — tạo lối tắt

> Chuột phải `visdrone-4` → **Sắp xếp** → **Thêm lối tắt vào Drive**
> → **Drive của tôi**

Lối tắt **không phải bản sao**: chủ sở hữu vẫn là người kia, không tốn dung
lượng Drive của bạn. Nó chỉ tạo một mục thật trong cây My Drive trỏ tới file
đó, để mount có đường dẫn mà đi tới. Để `ZIP_LINKS` trống thì cell tự dò.

In [ ]:
import os, sys, glob, subprocess

# ============================================================
#  CACH 0 (NHANH NHAT): ultralytics tu tai VisDrone-DET
# ============================================================
# Khong can upload gi, khong can Drive. Colab tai thang tu nguon, ~2 GB o toc
# do datacenter thay vi toc do upload tu nha. Duoc 6471 anh train / 548 val,
# dung bo VisDrone-DET voi 208 clip khac nhau.
#
# Da doi chieu bo chuyen doi cua ultralytics voi bo cua notebook nay tren
# 343.205 box that: ra ket qua Y HET. (No chi loc score=0 chu khong loc
# category 0/11, nhung trong VisDrone-DET hai loai do luon co score=0, nen
# khong sinh nhan sai nao.)
VISDRONE_AUTO = True

# ---- CACH A: dan link chia se vao day (khong dung toi Drive cua ban) ----
ZIP_LINKS = {
    "train": "",   # link chia se cua VisDrone2019-MOT-train.zip
    "val":   "",   # link chia se cua VisDrone2019-MOT-val.zip
}

# ---- CACH B: de trong ZIP_LINKS, cell se tu do trong MyDrive -----------
DATASET_DIR = ""   # dien tay neu muon chi dinh chinh xac

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".JPG", ".JPEG", ".PNG")
ZIP_TRAIN = ZIP_VAL = None
AUTO_ROOT = None

if VISDRONE_AUTO:
    from ultralytics.utils.downloads import safe_download
    from ultralytics.data.utils import check_det_dataset
    print("Tai VisDrone-DET (~2 GB, ultralytics tu chuyen sang YOLO) ...\n", flush=True)
    info = check_det_dataset("VisDrone.yaml")
    AUTO_ROOT = str(info["path"])
    for split in ("train", "val"):
        d = os.path.join(AUTO_ROOT, "images", split)
        n = len([f for f in os.listdir(d) if f.endswith(IMG_EXT)]) if os.path.isdir(d) else 0
        print(f"  {split:5s} {n:6d} anh   {d}")
    print(f"\n[cach 0] tu tai, khong dung Drive, khong upload gi")

elif all(ZIP_LINKS.values()):
    # subprocess chu khong phai %pip: magic nam trong khoi if la thu de vo,
    # va cach nay chay giong nhau du notebook duoc chay bang gi.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown>=5.1"],
                   check=True)
    import gdown
    os.makedirs("/content/zips", exist_ok=True)
    for split, url in ZIP_LINKS.items():
        out = f"/content/zips/VisDrone-MOT-{split}.zip"
        if os.path.exists(out) and os.path.getsize(out) > 1e8:
            print(f"[bo qua] {split}: da tai roi ({os.path.getsize(out)/1e9:.2f} GB)")
        else:
            print(f"Dang tai {split} ...", flush=True)
            # fuzzy: chap nhan link dang /file/d/<id>/view chu khong can tach id
            gdown.download(url=url, output=out, quiet=False, fuzzy=True)
        assert os.path.exists(out), (
            f"Tai {split} that bai. Thuong la do quyen chia se dang 'Bi han che' "
            f"-- doi sang 'Bat ky ai co duong lien ket', hoac dung cach B.")
    ZIP_TRAIN, ZIP_VAL = "/content/zips/VisDrone-MOT-train.zip", \
                         "/content/zips/VisDrone-MOT-val.zip"
    print("\n[cach A] tai truc tiep, khong dung toi Drive cua ban")

else:
    if not DATASET_DIR:
        print("Dang tim file VisDrone .zip trong MyDrive ...")
        hits = set()
        for depth in ("*", "*/*", "*/*/*"):
            for z in glob.glob(f"/content/drive/MyDrive/{depth}.zip"):
                if "visdrone" in os.path.basename(z).lower():
                    hits.add(os.path.dirname(z))
        hits = sorted(hits)
        if len(hits) == 1:
            DATASET_DIR = hits[0]
            print(f"  tim thay: {DATASET_DIR}")
        elif len(hits) > 1:
            for h in hits:
                print("   ", h)
            raise SystemExit("Nhieu noi khop -- dien mot cai vao DATASET_DIR.")

    assert DATASET_DIR and os.path.isdir(DATASET_DIR), (
        "Khong thay dataset trong MyDrive.\n"
        "'Duoc chia se voi toi' KHONG phai thu muc that nen Colab khong mount duoc.\n"
        "  -> Cach A: dan link chia se vao ZIP_LINKS o dau cell nay, hoac\n"
        "  -> Cach B: chuot phai thu muc -> Sap xep -> Them loi tat vao Drive\n"
        "             (loi tat KHONG phai ban sao, khong ton dung luong cua ban)")

    ZIPS = sorted(glob.glob(os.path.join(DATASET_DIR, "*.zip")))
    ZIP_TRAIN = next((z for z in ZIPS if "train" in os.path.basename(z).lower()), None)
    ZIP_VAL = next((z for z in ZIPS if "val" in os.path.basename(z).lower()), None)
    print(f"\n[cach B] doc qua loi tat trong MyDrive: {DATASET_DIR}")

if not VISDRONE_AUTO:
    assert ZIP_TRAIN and ZIP_VAL, f"Thieu zip: train={ZIP_TRAIN} val={ZIP_VAL}"
    for z in (ZIP_TRAIN, ZIP_VAL):
        print(f"  {os.path.basename(z):34s} {os.path.getsize(z)/1e9:5.2f} GB")
    print("\nKieu dataset nhan dang o cell 7 THEO CAU TRUC sau khi giai nen")
    print("(sequences/ = MOT, images/+annotations/ = DET), khong doan theo ten file.")

## 6. Giải nén về đĩa local

**Đừng train trực tiếp trên Drive.** Drive gắn qua FUSE — mỗi lần mở một file
ảnh là một lượt gọi mạng. Train ở batch 128 cần vài trăm ảnh/giây, FUSE không
đáp ứng nổi; GPU sẽ ngồi chờ và bạn tưởng model chậm trong khi thật ra là I/O.

Chép file `.zip` (một file lớn) rồi giải nén tại chỗ — nhanh hơn hẳn so với
chép hàng chục nghìn ảnh lẻ qua FUSE. Mỗi tab là một runtime riêng nên tab nào
cũng phải làm lại bước này.

In [ ]:
import os, glob, time, shutil, zipfile

LOCAL = "/content/dataset"
os.makedirs(LOCAL, exist_ok=True)

ZIPS_TO_EXTRACT = [] if VISDRONE_AUTO else [("train", ZIP_TRAIN), ("val", ZIP_VAL)]
if VISDRONE_AUTO:
    print(f"[bo qua] cach 0 da tai va giai nen san tai {AUTO_ROOT}")
    print(f"         cell 7 se cat o thang tu do.")

def already_extracted(split):
    """Thu muc giai nen mang ten trong zip (VisDrone2019-...), khong phai ten
    file zip, nen kiem tra theo noi dung chu dung theo ten."""
    for d in glob.glob(os.path.join(LOCAL, "*")):
        if os.path.isdir(d) and split in os.path.basename(d).lower() and (
                os.path.isdir(os.path.join(d, "sequences"))
                or os.path.isdir(os.path.join(d, "images"))):
            return d
    return None


for split, z in ZIPS_TO_EXTRACT:
    d = already_extracted(split)
    if d:
        print(f"[bo qua] {split} da giai nen: {os.path.basename(d)}")
        continue

    name = os.path.basename(z)
    t0 = time.time()

    # Chi chep khi zip con nam tren Drive (FUSE). Cach A da tai san ve dia
    # local roi -- chep them mot ban 4 GB nua la phi thoi gian va phi dia.
    on_drive = z.startswith("/content/drive")
    if on_drive:
        src = os.path.join("/content", name)
        print(f"Chep {name} ({os.path.getsize(z)/1e9:.2f} GB) tu Drive ...", flush=True)
        shutil.copy(z, src)
        print(f"  chep xong sau {time.time()-t0:.0f}s", flush=True)
    else:
        src = z
        print(f"{name} da o dia local, giai nen thang", flush=True)

    print("  dang giai nen ...", flush=True)
    with zipfile.ZipFile(src) as zf:
        zf.extractall(LOCAL)
    if on_drive:
        os.remove(src)
    print(f"  [ok] {split} sau {(time.time()-t0)/60:.1f} phut")

print("\nDa co trong /content/dataset:")
for d in sorted(os.listdir(LOCAL)):
    print("  ", d)
free = shutil.disk_usage("/content").free / 1e9
print(f"\nDia con trong: {free:.0f} GB")

## 7. Chuyển MOT → YOLO, **cắt ô chồng lấn**, và xáo trộn

### Vì sao phải chuyển định dạng

VisDrone-MOT lưu nhãn **một file cho cả sequence**, mỗi dòng 10 cột:

```
frame_index, target_id, x, y, w, h, score, category, truncation, occlusion
```

YOLO cần **một file cho mỗi ảnh**, 5 cột, toạ độ chuẩn hoá `[0,1]`. Ba chỗ dễ
sai thầm lặng, đều xử lý ở đây:

- **Chỉ số lớp lệch 1.** MOT đánh `1=pedestrian … 10=motor`; YOLO cần `0..9`.
- **`score = 0` là vùng bỏ qua**, không phải vật thể tin cậy thấp.
- **`category` 0 (ignored) và 11 (others)** nằm ngoài 10 lớp.

### Vì sao cắt ô — và cắt được bao nhiêu

Frame 1920×1080 letterbox về 640 là co lại **0.33×**. Người đi bộ 15px thành
5px, rồi mosaic ghép 4 ảnh co thêm 2× nữa còn **2.5px**. Tầng stride-8 không
biểu diễn nổi vật thể 2.5px — phần lớn nhãn đang dạy model một việc bất khả thi.

Cắt 2×2 chồng lấn 20%: mỗi ô 1067×600, letterbox gain **0.60** thay vì 0.33.

| | trước | sau khi cắt ô |
|---|---:|---:|
| người đi bộ 15px | 5.0 px | **9.0 px** |
| xe máy 25px | 8.3 px | **15.0 px** |
| ô tô 45px | 15.0 px | **27.0 px** |

Đúng **1.80×** cho mọi kích thước. Đổi lại: 4× số ảnh, 4× thời gian epoch, và
inference sau này phải chạy 4 lần rồi gộp — đúng cái đánh đổi bạn chấp nhận.

### Chồng lấn để làm gì

Vật thể nằm vắt qua đường cắt sẽ bị cụt ở cả hai ô nếu cắt sát nhau. Chồng lấn
20% đảm bảo mọi vật thể nhỏ hơn vùng chồng lấn đều **nguyên vẹn trong ít nhất
một ô**. Box bị cắt còn dưới `MIN_BOX_VISIBLE` thì loại — dạy model rằng một
mẩu ô tô là ô tô sẽ sinh ra false positive khắp nơi.

### Cái bẫy trong chữ "shuffle"

**Không được xáo trộn frame rồi chia lại train/val.** Frame liền nhau trong
video gần như trùng khớp; trộn xong chia ngẫu nhiên là để frame 100 vào train
và frame 101 vào val — model đã thấy gần đúng ảnh đó rồi, mAP val sẽ đẹp giả
tạo. Dataset của bạn đang chia **theo sequence** (2 zip riêng), đúng cách, và
notebook giữ nguyên như vậy.

Cái được xáo trộn là **thứ tự trong tập train**, ghi ra `train.txt`. Có ích
thật: nếu để nguyên thứ tự sinh ra, danh sách sắp theo sequence, và bất kỳ phép
lấy mẫu nào theo tiền tố (như `fraction=0.5` của ultralytics) sẽ lấy trúng nửa
số cảnh thay vì một nửa trải đều.

In [ ]:
import os, glob, yaml, random, collections, shutil, json, math, cv2
import numpy as np
from concurrent.futures import ThreadPoolExecutor

FRAME_STRIDE = 3          # 1 = lay moi frame

# MOT category -> chi so YOLO. Bo 0 (ignored) va 11 (others).
MOT2YOLO = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4, 6: 5, 7: 6, 8: 7, 9: 8, 10: 9}
VISDRONE_NAMES = ["pedestrian", "people", "bicycle", "car", "van", "truck",
                  "tricycle", "awning-tricycle", "bus", "motor"]

YOLO_ROOT = os.path.join(LOCAL, "yolo")


def tile_boxes(W, H, grid, ov):
    """Toa do cac o. tw*(grid - ov) = W  ->  tw = W / (grid - ov)."""
    tw, th = W / (grid - ov), H / (grid - ov)
    xs = [round((W - tw) * i / (grid - 1)) for i in range(grid)] if grid > 1 else [0]
    ys = [round((H - th) * i / (grid - 1)) for i in range(grid)] if grid > 1 else [0]
    return [(x, y, int(round(tw)), int(round(th))) for y in ys for x in xs]


def clip_to_tile(boxes, tx, ty, tw, th, min_vis):
    """boxes: [(cls, x1, y1, x2, y2)] pixel toan anh -> nhan YOLO trong o.

    Box vat qua canh o duoc cat lai, va chi giu neu con >= min_vis dien tich.
    Bo qua nguong nay thi mot mep xe cung thanh mot nhan "xe", va model hoc
    cach bao xe o moi noi co mot manh kim loai."""
    out = []
    for cls, x1, y1, x2, y2 in boxes:
        ix1, iy1 = max(x1, tx), max(y1, ty)
        ix2, iy2 = min(x2, tx + tw), min(y2, ty + th)
        if ix2 <= ix1 or iy2 <= iy1:
            continue
        area = (x2 - x1) * (y2 - y1)
        if area <= 0 or (ix2 - ix1) * (iy2 - iy1) / area < min_vis:
            continue
        cx, cy = (ix1 + ix2) / 2 - tx, (iy1 + iy2) / 2 - ty
        bw, bh = ix2 - ix1, iy2 - iy1
        out.append(f"{cls} {cx/tw:.6f} {cy/th:.6f} {bw/tw:.6f} {bh/th:.6f}")
    return out


def _link(src, dst):
    """Symlink de khong nhan doi vai GB anh; hardlink roi copy la duong lui."""
    if os.path.exists(dst):
        return
    for fn in (os.symlink, os.link, shutil.copy):
        try:
            fn(src, dst)
            return
        except (OSError, NotImplementedError, AttributeError):
            continue


def find_root(local, want):
    """Tim thu muc cua mot split va nhan dang kieu dataset THEO CAU TRUC.

    Nhan theo ten file zip la de sai: nguoi khac doi ten la hong. Cau truc thi
    khong noi doi:
      sequences/ + annotations/  -> MOT (nhan 10 cot, 1 file / ca sequence)
      images/    + annotations/  -> DET (nhan  8 cot, 1 file / anh)
      images/    + labels/       -> da la YOLO
    """
    for d in sorted(glob.glob(os.path.join(local, "*"))):
        if not os.path.isdir(d) or want not in os.path.basename(d).lower():
            continue
        has_ann = os.path.isdir(os.path.join(d, "annotations"))
        if os.path.isdir(os.path.join(d, "sequences")) and has_ann:
            return d, "MOT"
        if os.path.isdir(os.path.join(d, "images")) and has_ann:
            return d, "DET"
        if os.path.isdir(os.path.join(d, "images")) and os.path.isdir(
                os.path.join(d, "labels")):
            return d, "YOLO"
    return None, None


def convert_det(det_root, split, stride=1):
    """VisDrone-DET -> YOLO, cung duong cat o nhu MOT.

    Nhan DET co 8 cot, mot file cho moi anh:
        x, y, w, h, score, category, truncation, occlusion
    Khong co frame_index va target_id o dau nhu MOT -- do la khac biet duy nhat
    ve dinh dang. Quy tac loai bo giong het: score=0 la vung bo qua,
    category 0/11 nam ngoai 10 lop, class id lech 1.
    """
    tile_img = os.path.join(YOLO_ROOT, split, "images")
    tile_lab = os.path.join(YOLO_ROOT, split, "labels")
    full_img = os.path.join(YOLO_ROOT, split + "_full", "images")
    full_lab = os.path.join(YOLO_ROOT, split + "_full", "labels")
    for d in (tile_img, tile_lab, full_img, full_lab):
        os.makedirs(d, exist_ok=True)

    img_dir = os.path.join(det_root, "images")
    ann_dir = os.path.join(det_root, "annotations")
    files = sorted(f for f in os.listdir(img_dir) if f.endswith(IMG_EXT))
    files = files[::stride]
    stat = collections.Counter()
    listing, issues = [], []

    def do_one(fname):
        stem = os.path.splitext(fname)[0]
        ann = os.path.join(ann_dir, stem + ".txt")
        src = os.path.join(img_dir, fname)
        im = cv2.imread(src)
        if im is None:
            return [], 1, 0
        H, W = im.shape[:2]

        raw = []
        if os.path.exists(ann):
            for line in open(ann):
                p = line.strip().rstrip(",").split(",")
                if len(p) < 6:
                    continue
                x, y, w, h, score, cat = (int(float(v)) for v in p[:6])
                if score == 0:
                    stat["bo_vung_ignored"] += 1
                    continue
                if cat not in MOT2YOLO:
                    stat["bo_lop_0_11"] += 1
                    continue
                x1, y1 = max(0, x), max(0, y)
                x2, y2 = min(W, x + w), min(H, y + h)
                if x2 > x1 and y2 > y1:
                    raw.append((MOT2YOLO[cat], x1, y1, x2, y2))

        _link(src, os.path.join(full_img, fname))
        with open(os.path.join(full_lab, stem + ".txt"), "w") as f:
            f.write("\n".join(
                f"{c} {(x1+x2)/2/W:.6f} {(y1+y2)/2/H:.6f} "
                f"{(x2-x1)/W:.6f} {(y2-y1)/H:.6f}" for c, x1, y1, x2, y2 in raw))

        if not TILE:
            _link(src, os.path.join(tile_img, fname))
            shutil.copy(os.path.join(full_lab, stem + ".txt"),
                        os.path.join(tile_lab, stem + ".txt"))
            return [os.path.join(tile_img, fname)], 0, len(raw)

        paths, nb = [], 0
        for ti, (tx, ty, tw, th) in enumerate(tile_boxes(W, H, TILE_GRID, TILE_OVERLAP)):
            lines = clip_to_tile(raw, tx, ty, tw, th, MIN_BOX_VISIBLE)
            tname = f"{stem}_t{ti:02d}"
            p = os.path.join(tile_img, tname + ".jpg")
            if not os.path.exists(p):
                cv2.imwrite(p, im[ty:ty + th, tx:tx + tw],
                            [cv2.IMWRITE_JPEG_QUALITY, 95])
            with open(os.path.join(tile_lab, tname + ".txt"), "w") as f:
                f.write("\n".join(lines))
            paths.append(p)
            nb += len(lines)
        return paths, 0, nb

    with ThreadPoolExecutor(max_workers=8) as ex:
        for k, (paths, bad, nb) in enumerate(ex.map(do_one, files)):
            listing.extend(paths)
            stat["anh_loi"] += bad
            stat["box"] += nb
            if (k + 1) % 1000 == 0:
                print(f"    {k+1}/{len(files)} anh, {len(listing)} o", flush=True)

    return listing, stat, issues


def convert_mot(mot_root, split, stride):
    """MOT -> YOLO. Viet ca ban cat o va ban nguyen khung hinh.

    Ban nguyen khung (<split>_full) duoc giu lai de cell 16 do duoc chat luong
    cua inference cat-o-roi-gop tren anh that -- do moi la con so trien khai.
    """
    tile_img = os.path.join(YOLO_ROOT, split, "images")
    tile_lab = os.path.join(YOLO_ROOT, split, "labels")
    full_img = os.path.join(YOLO_ROOT, split + "_full", "images")
    full_lab = os.path.join(YOLO_ROOT, split + "_full", "labels")
    for d in (tile_img, tile_lab, full_img, full_lab):
        os.makedirs(d, exist_ok=True)

    seq_dirs = sorted(glob.glob(os.path.join(mot_root, "sequences", "*")))
    stat = collections.Counter()
    listing = []
    issues = []

    def do_frame(args):
        seq, name, fname, by_frame, W, H = args
        fi = int(os.path.splitext(fname)[0])
        stem = f"{name}_{fi:07d}"
        raw = by_frame.get(fi, [])

        # --- ban nguyen khung: symlink, khong ton dia ---
        _link(os.path.join(seq, fname),
              os.path.join(full_img, stem + os.path.splitext(fname)[1]))
        with open(os.path.join(full_lab, stem + ".txt"), "w") as f:
            f.write("\n".join(
                f"{c} {(x1+x2)/2/W:.6f} {(y1+y2)/2/H:.6f} "
                f"{(x2-x1)/W:.6f} {(y2-y1)/H:.6f}" for c, x1, y1, x2, y2 in raw))

        if not TILE:
            _link(os.path.join(seq, fname),
                  os.path.join(tile_img, stem + os.path.splitext(fname)[1]))
            shutil.copy(os.path.join(full_lab, stem + ".txt"),
                        os.path.join(tile_lab, stem + ".txt"))
            return [os.path.join(tile_img, stem + os.path.splitext(fname)[1])], 0, len(raw)

        im = cv2.imread(os.path.join(seq, fname))
        if im is None:
            return [], 1, 0
        paths, nb = [], 0
        for ti, (tx, ty, tw, th) in enumerate(tile_boxes(W, H, TILE_GRID, TILE_OVERLAP)):
            lines = clip_to_tile(raw, tx, ty, tw, th, MIN_BOX_VISIBLE)
            tname = f"{stem}_t{ti:02d}"
            p = os.path.join(tile_img, tname + ".jpg")
            if not os.path.exists(p):
                cv2.imwrite(p, im[ty:ty + th, tx:tx + tw],
                            [cv2.IMWRITE_JPEG_QUALITY, 95])
            with open(os.path.join(tile_lab, tname + ".txt"), "w") as f:
                f.write("\n".join(lines))
            paths.append(p)
            nb += len(lines)
        return paths, 0, nb

    for si, seq in enumerate(seq_dirs):
        name = os.path.basename(seq)
        ann = os.path.join(mot_root, "annotations", name + ".txt")
        if not os.path.exists(ann):
            issues.append(f"{name}: khong co annotation")
            continue
        frames = sorted(f for f in os.listdir(seq) if f.endswith(IMG_EXT))
        if not frames:
            continue
        probe = cv2.imread(os.path.join(seq, frames[0]))
        if probe is None:
            issues.append(f"{name}: khong doc duoc frame dau")
            continue
        H, W = probe.shape[:2]

        by_frame = collections.defaultdict(list)
        for line in open(ann):
            p = line.strip().split(",")
            if len(p) < 8:
                continue
            fi, _tid, x, y, w, h, score, cat = (int(float(v)) for v in p[:8])
            if score == 0:
                stat["bo_vung_ignored"] += 1
                continue
            if cat not in MOT2YOLO:
                stat["bo_lop_0_11"] += 1
                continue
            x1, y1 = max(0, x), max(0, y)
            x2, y2 = min(W, x + w), min(H, y + h)
            if x2 <= x1 or y2 <= y1:
                continue
            by_frame[fi].append((MOT2YOLO[cat], x1, y1, x2, y2))

        jobs = [(seq, name, f, by_frame, W, H)
                for k, f in enumerate(frames) if k % stride == 0]
        with ThreadPoolExecutor(max_workers=8) as ex:
            for paths, bad, nb in ex.map(do_frame, jobs):
                listing.extend(paths)
                stat["anh_loi"] += bad
                stat["box"] += nb
                stat["anh"] += 1 if paths else 0

        if (si + 1) % 10 == 0:
            print(f"    {si+1}/{len(seq_dirs)} sequence, {len(listing)} anh",
                  flush=True)

    return listing, stat, issues


def convert_yolo(img_dir, lab_dir, split):
    """Da o dang YOLO roi (vd ban ultralytics tu tai ve) -> chi cat o.

    Nhan YOLO da chuan hoa theo anh goc, nen phai doi nguoc ve pixel truoc khi
    cat, roi chuan hoa lai theo kich thuoc o. Bo qua buoc nay thi box van "hop
    le" ([0,1]) nhung nam sai cho hoan toan -- khong metric nao bao duoc.
    """
    tile_img = os.path.join(YOLO_ROOT, split, "images")
    tile_lab = os.path.join(YOLO_ROOT, split, "labels")
    full_img = os.path.join(YOLO_ROOT, split + "_full", "images")
    full_lab = os.path.join(YOLO_ROOT, split + "_full", "labels")
    for d in (tile_img, tile_lab, full_img, full_lab):
        os.makedirs(d, exist_ok=True)

    files = sorted(f for f in os.listdir(img_dir) if f.endswith(IMG_EXT))
    stat = collections.Counter()
    listing, issues = [], []

    def do_one(fname):
        stem = os.path.splitext(fname)[0]
        src = os.path.join(img_dir, fname)
        im = cv2.imread(src)
        if im is None:
            return [], 1, 0
        H, W = im.shape[:2]

        raw = []
        lf = os.path.join(lab_dir, stem + ".txt")
        if os.path.exists(lf):
            for line in open(lf):
                p = line.split()
                if len(p) < 5:
                    continue
                c = int(float(p[0]))
                cx, cy, bw, bh = (float(v) for v in p[1:5])
                if not (0 <= c < len(VISDRONE_NAMES)):
                    stat["bo_lop_ngoai_pham_vi"] += 1
                    continue
                x1, y1 = (cx - bw / 2) * W, (cy - bh / 2) * H
                x2, y2 = (cx + bw / 2) * W, (cy + bh / 2) * H
                x1, y1 = max(0.0, x1), max(0.0, y1)
                x2, y2 = min(float(W), x2), min(float(H), y2)
                if x2 > x1 and y2 > y1:
                    raw.append((c, x1, y1, x2, y2))

        _link(src, os.path.join(full_img, fname))
        shutil.copy(lf, os.path.join(full_lab, stem + ".txt")) if os.path.exists(lf) \
            else open(os.path.join(full_lab, stem + ".txt"), "w").close()

        if not TILE:
            _link(src, os.path.join(tile_img, fname))
            shutil.copy(os.path.join(full_lab, stem + ".txt"),
                        os.path.join(tile_lab, stem + ".txt"))
            return [os.path.join(tile_img, fname)], 0, len(raw)

        paths, nb = [], 0
        for ti, (tx, ty, tw, th) in enumerate(tile_boxes(W, H, TILE_GRID, TILE_OVERLAP)):
            lines = clip_to_tile(raw, tx, ty, tw, th, MIN_BOX_VISIBLE)
            tname = f"{stem}_t{ti:02d}"
            p = os.path.join(tile_img, tname + ".jpg")
            if not os.path.exists(p):
                cv2.imwrite(p, im[ty:ty + th, tx:tx + tw],
                            [cv2.IMWRITE_JPEG_QUALITY, 95])
            with open(os.path.join(tile_lab, tname + ".txt"), "w") as f:
                f.write("\n".join(lines))
            paths.append(p)
            nb += len(lines)
        return paths, 0, nb

    with ThreadPoolExecutor(max_workers=8) as ex:
        for k, (paths, bad, nb) in enumerate(ex.map(do_one, files)):
            listing.extend(paths)
            stat["anh_loi"] += bad
            stat["box"] += nb
            if (k + 1) % 1000 == 0:
                print(f"    {k+1}/{len(files)} anh, {len(listing)} o", flush=True)

    return listing, stat, issues


splits, listings = {}, {}
names, NC = VISDRONE_NAMES, len(VISDRONE_NAMES)
print("cat o: " + (f"{TILE_GRID}x{TILE_GRID} chong lan {int(TILE_OVERLAP*100)}%"
                   if TILE else "khong") + "\n")

for split in ("train", "val"):
    if VISDRONE_AUTO:
        root, kind = AUTO_ROOT, "YOLO_SPLIT"
        print(f"  [{split}] {AUTO_ROOT}  (YOLO, ultralytics tu tai)")
    else:
        root, kind = find_root(LOCAL, split)
        assert root, (f"khong thay du lieu cho split {split} trong {LOCAL}. "
                      f"Co: {[os.path.basename(d) for d in glob.glob(LOCAL + '/*')]}")
        print(f"  [{split}] {os.path.basename(root)}  ({kind})")

    if kind == "YOLO_SPLIT":
        lst, stat, iss = convert_yolo(os.path.join(root, "images", split),
                                      os.path.join(root, "labels", split), split)
    elif kind == "MOT":
        # FRAME_STRIDE chi co nghia voi video. Anh DET la anh roi, lay het.
        lst, stat, iss = convert_mot(root, split, FRAME_STRIDE)
        print(f"    stride {FRAME_STRIDE}")
    elif kind == "DET":
        lst, stat, iss = convert_det(root, split, 1)
    else:
        raise SystemExit(f"{root} da o dang YOLO -- bo qua buoc chuyen doi, "
                         f"tro thang data.yaml vao no.")

    print(f"    {len(lst)} anh, {stat['box']} box  "
          f"(bo {stat['bo_vung_ignored']} vung ignored, "
          f"{stat['bo_lop_0_11']} box lop 0/11)")
    for x in iss[:5]:
        print("    !!", x)
    listings[split] = lst
    splits[split] = os.path.join(YOLO_ROOT, split, "images")

assert splits.get("train") and splits.get("val"), f"thieu split: {splits}"

# ---- Xao tron thu tu tap train, ghi ra .txt -----------------------------
# Chi xao tron THU TU BEN TRONG train. Khong dong den viec chia train/val:
# frame lien nhau trong video gan nhu trung khop, tron roi chia lai la de
# model thay truoc anh val -> mAP dep gia tao.
LIST_TXT = {}
for split in ("train", "val"):
    paths = sorted(listings[split])
    if split == "train":
        random.Random(SHUFFLE_SEED).shuffle(paths)
    p = os.path.join(YOLO_ROOT, f"{split}.txt")
    with open(p, "w") as f:
        f.write("\n".join(paths) + "\n")
    LIST_TXT[split] = p
    print(f"  {split}.txt : {len(paths)} anh"
          f"{'  (da xao tron, seed ' + str(SHUFFLE_SEED) + ')' if split == 'train' else ''}")

print(f"\nnc = {NC}: {names}")

# ---- bon kiem tra ------------------------------------------------------
problems, hist = [], collections.Counter()
for split, img_dir in splits.items():
    lab_dir = os.path.join(os.path.dirname(img_dir), "labels")
    if not os.path.isdir(lab_dir):
        problems.append(f"{split}: khong thay {lab_dir}")
        continue

    imgs = [f for f in os.listdir(img_dir) if f.endswith(IMG_EXT)]
    missing = sum(1 for f in imgs if not os.path.exists(
        os.path.join(lab_dir, os.path.splitext(f)[0] + ".txt")))
    if missing:
        problems.append(f"{split}: {missing}/{len(imgs)} anh khong co file label")

    lfs = glob.glob(os.path.join(lab_dir, "*.txt"))
    bad_range = bad_cls = n_box = n_empty = 0
    for lf in random.Random(0).sample(lfs, min(500, len(lfs))):
        txt = open(lf).read().strip()
        if not txt:
            n_empty += 1
            continue
        for line in txt.split("\n"):
            p = line.split()
            if len(p) < 5:
                continue
            n_box += 1
            c = int(float(p[0]))
            hist[c] += 1
            if not (0 <= c < NC):
                bad_cls += 1
            if any(not (0.0 <= float(v) <= 1.0) for v in p[1:5]):
                bad_range += 1
    if bad_range:
        problems.append(f"{split}: {bad_range}/{n_box} box ngoai [0,1]")
    if bad_cls:
        problems.append(f"{split}: {bad_cls}/{n_box} box co class id ngoai [0,{NC})")
    print(f"  {split:5s} {len(imgs):6d} anh | {n_empty}/500 mau khong co vat the")

# O nao khong chua vat the nao van co ich (anh nen), nhung qua nhieu thi lam
# loang tin hieu. Bao ra de biet chu khong tu dong loai.
if TILE:
    empty_frac = n_empty / 500
    if empty_frac > 0.5:
        problems.append(f"hon {empty_frac*100:.0f}% o khong co vat the nao -- "
                        f"can nhac giam TILE_GRID hoac tang FRAME_STRIDE")

print("\nPhan bo lop (mau 500 file/split):")
mx = max(hist.values()) if hist else 1
for c in range(NC):
    print(f"  {c:2d} {names[c]:18s} {hist[c]:7d} {'#' * int(40 * hist[c] / mx)}")
    if hist[c] == 0:
        problems.append(f"lop {c} ({names[c]}) khong co box nao -> AP lop nay = -1")

# --- So box cua anh DAY NHAT: con so quyet dinh batch toi da -------------
# TaskAlignedAssigner cap phat tensor (bs, n_max_boxes, n_anchors) -- xem
# ultralytics/utils/tal.py. Cat o chia nho so vat the moi anh nen con so nay
# giam han, va batch cho phep tang len theo.
_train_lfs = glob.glob(os.path.join(os.path.dirname(splits["train"]), "labels", "*.txt"))
_counts = []
for lf in _train_lfs:
    with open(lf) as f:
        _counts.append(sum(1 for l in f if l.strip()))
_counts.sort()
MAX_BOXES = max(1, _counts[-1] if _counts else 1)
print(f"\nSo vat the tren mot anh (train, {len(_counts)} anh):")
print(f"  trung binh {sum(_counts)/max(len(_counts),1):6.1f}")
print(f"  p99        {_counts[int(len(_counts)*0.99)] if _counts else 0:6d}")
print(f"  lon nhat   {MAX_BOXES:6d}   <- con so chan batch o cell 9")

DATA_YAML = os.path.join(os.path.dirname(LOCAL), "data.yaml")
yaml.safe_dump({"path": LOCAL, "train": LIST_TXT["train"], "val": LIST_TXT["val"],
                "nc": NC, "names": names},
               open(DATA_YAML, "w"), sort_keys=False, allow_unicode=True)

# data.yaml rieng cho anh nguyen khung, cell 16 dung de do inference cat-o.
DATA_YAML_FULL = os.path.join(os.path.dirname(LOCAL), "data_full.yaml")
yaml.safe_dump({"path": LOCAL,
                "train": os.path.join(YOLO_ROOT, "train_full", "images"),
                "val": os.path.join(YOLO_ROOT, "val_full", "images"),
                "nc": NC, "names": names},
               open(DATA_YAML_FULL, "w"), sort_keys=False, allow_unicode=True)

print("\n" + "=" * 62)
if problems:
    print("VAN DE - doc ky truoc khi train:")
    for p in problems:
        print("  !!", p)
else:
    print("Khong phat hien van de nao.")
print("=" * 62)
print(open(DATA_YAML).read())

## 8. Preflight — dựng model trước khi train

Ba tiếng train rồi mới biết model không dựng được là ba tiếng mất trắng. Cell
này dựng model, in số tham số, số anchor, và **tỉ lệ trọng số nạp được** —
con số cuối chính là bằng chứng cho caveat của p2, nên nó phải hiện ra chứ
không được im lặng.

In [ ]:
import warnings, io, contextlib, torch
warnings.filterwarnings("ignore")
from ultralytics import YOLO

buf = io.StringIO()
with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
    _m = YOLO(SPEC["cfg"], verbose=False)
    _m.load(SPEC["weights"])
    _src = YOLO(SPEC["weights"]).model.state_dict()

# Dem truc tiep tren state_dict thay vi doc log: ultralytics in dong
# "Transferred x/y" qua LOGGER rieng, redirect_stdout khong bat duoc -- va mot
# cot im lang bao "n/a" chinh la cot khong ai kiem tra. Day dung la tieu chi
# ultralytics dung khi nap: trung ten VA trung shape.
_dst = _m.model.state_dict()
MATCHED = sum(1 for k, v in _dst.items()
              if k in _src and _src[k].shape == v.shape)
TRANSFER = f"{MATCHED}/{len(_dst)}"
TRANSFER_FRAC = MATCHED / len(_dst)

_head = _m.model.model[-1]
END2END = bool(getattr(_head, "end2end", False))
if END2END:
    _head.max_det = MAX_DET

_m.model.eval()
with torch.no_grad():
    _y = _m.model(torch.zeros(1, 3, IMGSZ, IMGSZ))
_out = _y[0] if isinstance(_y, (list, tuple)) else _y

N_PARAMS = sum(p.numel() for p in _m.model.parameters())
N_ANCHORS = None if END2END else int(_out.shape[-1])
# Preflight chay o nc=80 mac dinh cua yaml (model chua gap dataset), nen shape
# do duoc bay gio khong phai shape cuoi. Ghi ca hai de khong ai doc nham.
OUT_SHAPE_TRAINED = (1, MAX_DET, 6) if END2END else (1, 4 + NC, N_ANCHORS)

print(f"model            {MODEL}")
print(f"tham so          {N_PARAMS/1e6:.2f} M")
print(f"anchor           {N_ANCHORS if N_ANCHORS else '(end2end, khong dung anchor grid)'}")
print(f"NMS              {'KHONG can (end-to-end)' if END2END else 'can'}")
print(f"max_det          {MAX_DET if END2END else '(NMS quyet dinh)'}")
print(f"weights nap      {TRANSFER}  ({TRANSFER_FRAC*100:.0f}%)")
print(f"output sau train {OUT_SHAPE_TRAINED}")
print(f"  (preflight do duoc {tuple(_out.shape)} vi chay o nc=80 mac dinh cua yaml;")
print(f"   shape that duoc ghi lai tu file ONNX o cell 12)")

# Nguong 0.75 chu khong phai 0.9: model base thuong chi dat ~90% vi dau detect
# khong khop khi nc khac COCO -- do la binh thuong. Chi truong hop mat ca
# backbone/neck moi dang canh bao.
if TRANSFER_FRAC < 0.75:
    print(f"\n[luu y] Mot phan lon model khoi tao ngau nhien ({TRANSFER}).")
    print(f"        Da bu bang {SPEC['epochs']} epoch (cao hon {EPOCHS_BASE} cua cac model kia),")
    print(f"        nhung bang so sanh cuoi VAN phai ghi ro con so nay -- neu khong,")
    print(f"        nguoi doc se ket luan sai ve kien truc.")

del _m, _src, _dst
torch.cuda.empty_cache()

## 9. Đo bộ nhớ thật rồi tự chỉnh batch

`batch` ở cell 1 là ước lượng cho A100 40 GB. Cell này **đo thật** — vài bước
forward + backward với dữ liệu ngẫu nhiên rồi đọc đỉnh bộ nhớ — và tự **tăng**
nếu còn nhiều chỗ trống hoặc **giảm** nếu tràn.

Đây là cận dưới (chưa tính EMA và optimizer state, vốn nhỏ với model nano),
nhưng nó bắt OOM ở phút thứ hai thay vì giờ thứ hai.

In [ ]:
import torch, gc, warnings
warnings.filterwarnings("ignore")
from ultralytics import YOLO


def _all_tensors(o):
    """Dau train tra ve dict (yolo26 tra one2many/one2one), khong phai list.
    Gom het tensor lai roi tinh mot loss gia -- chi de do bo nho."""
    if torch.is_tensor(o):
        return [o]
    if isinstance(o, dict):
        o = list(o.values())
    if isinstance(o, (list, tuple)):
        out = []
        for x in o:
            out.extend(_all_tensors(x))
        return out
    return []


def peak_mem_gb(cfg, batch, imgsz=IMGSZ, steps=3):
    torch.cuda.empty_cache(); gc.collect()
    torch.cuda.reset_peak_memory_stats()
    m = YOLO(cfg, verbose=False).model.cuda().train()
    opt = torch.optim.SGD(m.parameters(), lr=1e-4)
    scaler = torch.amp.GradScaler("cuda")
    try:
        for _ in range(steps):
            x = torch.rand(batch, 3, imgsz, imgsz, device="cuda")
            with torch.amp.autocast("cuda"):
                ts = [t for t in _all_tensors(m(x)) if t.is_floating_point()]
                if not ts:
                    raise RuntimeError("khong lay duoc tensor nao tu dau ra")
                loss = sum((t.float() ** 2).mean() for t in ts)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        peak = torch.cuda.max_memory_allocated() / 1e9
    except torch.cuda.OutOfMemoryError:
        peak = float("inf")
    finally:
        del m, opt
        torch.cuda.empty_cache(); gc.collect()
    return peak


BUDGET = VRAM_GB * 0.85     # chua 15% cho fragmentation va cudnn workspace
print(f"VRAM {VRAM_GB:.1f} GB -> ngan sach {BUDGET:.1f} GB\n")

# --- Tran do TaskAlignedAssigner, thuong chat hon ca activation ----------
# peak_mem_gb() o tren chi do activation cua model: no chay forward+backward
# voi mot loss gia, nen KHONG he cham toi ham loss that. Ma voi anh dong vat
# the thi chinh ham loss moi la thu an VRAM: TaskAlignedAssigner tao tensor
# (bs, n_max_boxes, n_anchors) va vai tensor cung co, xem tal.py.
#
# Bo qua dieu nay thi trieu chung khong phai crash ma la
#   "WARNING CUDA OutOfMemoryError in TaskAlignedAssigner, using CPU"
# lap lai moi batch -- ultralytics tu lui ve CPU nen train VAN CHAY, chi la
# cham gap ~15 lan. De nguyen se ngoi doi vai tieng ma khong hieu vi sao.
_na = N_ANCHORS if N_ANCHORS else sum((IMGSZ // s) ** 2 for s in (4, 8, 16, 32))
_BYTES = 4
_NTENSOR = 6            # mask_gt, align_metric, overlaps, mask_topk, mask_pos, tam
_ASSIGNER_GB = 4.0      # phan VRAM danh cho assigner

cap = int(_ASSIGNER_GB * 1e9 / (MAX_BOXES * _na * _BYTES * _NTENSOR))
cap = max(4, cap)
print(f"  anh day nhat {MAX_BOXES} vat the, {_na} anchor")
print(f"  -> tran assigner: batch <= {cap}")

want = BATCH_OVERRIDE if BATCH_OVERRIDE else SPEC["batch"]
batch = min(want, cap)
if BATCH_OVERRIDE:
    print(f"  BATCH_OVERRIDE = {BATCH_OVERRIDE} (dat cung cho ca 4 tab)")
if batch < want:
    print(f"  -> ha batch {want} xuong {batch} vi tran assigner")

peak = peak_mem_gb(SPEC["cfg"], batch)
print(f"\n  batch {batch:4d} -> activation dinh {peak:5.1f} GB")

while peak + _ASSIGNER_GB > BUDGET and batch > 4:
    batch = max(4, batch // 2)
    peak = peak_mem_gb(SPEC["cfg"], batch)
    print(f"  ha xuong {batch:4d} -> activation dinh {peak:5.1f} GB")

# Khong tu tang batch len nua. Lan truoc co doan tu nhan doi batch khi thay
# con trong VRAM, va no day batch len 512 -- vua du cho activation, nhung
# assigner thi tran, roi lui ve CPU va epoch cham gap 15 lan. Do thieu mot
# nua roi tu dong tang la cach chac chan nhat de sai.

SPEC["batch"] = batch
print(f"\n[chot] batch = {batch}")
print(f"       activation {peak:.1f} GB + assigner ~{_ASSIGNER_GB:.1f} GB "
      f"= ~{peak + _ASSIGNER_GB:.1f} / {BUDGET:.1f} GB")

if BATCH_OVERRIDE and batch < BATCH_OVERRIDE:
    print(f"\n  !!! TAB NAY KHONG CHAY DUOC BATCH {BATCH_OVERRIDE}, phai ha xuong {batch}.")
    print(f"      De 4 model con so sanh duoc voi nhau, dat")
    print(f"          BATCH_OVERRIDE = {batch}")
    print(f"      o CA BON tab roi train lai het. Giu nguyen moi tab mot batch")
    print(f"      khac nhau thi bang mAP cuoi cung khong ket luan duoc gi.")
elif BATCH_OVERRIDE:
    print(f"       khop BATCH_OVERRIDE -> tab nay so sanh duoc voi cac tab khac")

print(f"\nNeu khi train van thay 'OutOfMemoryError in TaskAlignedAssigner':")
print(f"  do la canh bao chu khong phai crash -- ultralytics lui ve CPU va train")
print(f"  cham gap ~15 lan. Ha BATCH_OVERRIDE xuong {max(4, batch // 2)} o ca 4 tab.")

## 10. Train

`project` trỏ **thẳng vào Drive** là có chủ ý: `/content` bị xoá sạch khi Colab
ngắt kết nối, nên checkpoint để ở đó thì mất trắng nhiều giờ. Để trên Drive thì
`last.pt` sống sót, và chạy lại đúng cell này là **tự resume**.

Dataset vẫn nằm ở `/content` (nhanh); chỉ checkpoint đi Drive. Đó là chỗ phân
chia đúng: ảnh đọc mỗi bước, checkpoint ghi mỗi epoch.

Augment: giữ photometric, **hạn chế geometric mạnh** — vật thể VisDrone rất nhỏ,
`scale`/`shear` lớn sẽ xoá sạch các box dưới 10 px.

In [ ]:
import os, glob, warnings, psutil
warnings.filterwarnings("ignore")
from ultralytics import YOLO

PROJECT = "/content/drive/MyDrive/skysentry/finetune_runs"
os.makedirs(PROJECT, exist_ok=True)

# --- Chon cache: don bay lon nhat ve toc do voi model nano ---------------
# Model nano tren A100 KHONG bi nghen o GPU. Tinh thu: yolov8n ~8.7 GFLOPs/anh
# o 640, train ~3x forward, tuc ~26 GFLOPs/anh. A100 chay fp16 thuc te vai chuc
# TFLOPS -> phan GPU cua mot epoch 6.5k anh chi vai giay. Cai an het thoi gian
# la giai nen JPEG tren CPU: mosaic doc 4 anh cho moi mau, batch 128 nghia la
# 512 lan decode anh 1920x1080 cho mot buoc, tren 12 vCPU cua Colab.
#
# cache="ram" giai nen mot lan roi giu anh da resize trong RAM -> vong lap
# khong con decode nua. Day thuong la khac biet 2-3x, lon hon moi thu khac
# trong cell nay cong lai.
_n_train = len([f for f in os.listdir(splits["train"]) if f.endswith(IMG_EXT)])
_ram_need = _n_train * IMGSZ * IMGSZ * 3 / 1e9        # can tren: anh vuong
_ram_free = psutil.virtual_memory().available / 1e9
_cpu = os.cpu_count() or 8

if _ram_need < _ram_free * 0.5:
    CACHE = "ram"
    _why = f"can ~{_ram_need:.1f} GB, con trong {_ram_free:.1f} GB"
elif _ram_need < 60:
    CACHE = "disk"
    _why = f"can ~{_ram_need:.1f} GB > 50% RAM trong ({_ram_free:.1f} GB) -> dung .npy tren dia"
else:
    CACHE = False
    _why = f"dataset qua lon ({_ram_need:.1f} GB)"

WORKERS = min(8, max(2, _cpu - 2))

print(f"anh train  : {_n_train}")
print(f"cache      : {CACHE}   ({_why})")
print(f"workers    : {WORKERS}  (thay {_cpu} vCPU)")
if CACHE == "ram":
    print("             -> epoch dau cham hon vi phai nap cache, cac epoch sau nhanh han")
print()

last = os.path.join(PROJECT, MODEL, "weights", "last.pt")
resume = False

if os.path.exists(last):
    import torch
    ck = torch.load(last, map_location="cpu", weights_only=False)
    prev = ck.get("train_args") or {}
    resumable = ck.get("optimizer") is not None and ck.get("epoch", -1) >= 0

    # Ultralytics khi resume se KHOI PHUC LAI toan bo args tu checkpoint va bo
    # qua moi tham so truyen vao lan nay. Nen neu cau hinh da doi (batch, cat o,
    # so epoch...), resume se lang le train tiep cau hinh CU -- ban tuong dang
    # chay cai moi ma khong phai. Doi chieu truoc, khong resume mu.
    want = {"batch": SPEC["batch"], "imgsz": IMGSZ,
            "epochs": SPEC["epochs"], "data": DATA_YAML}
    drift = {k: (prev.get(k), v) for k, v in want.items() if prev.get(k) != v}

    if not resumable:
        print(f"[moi] {last} khong con optimizer state (run truoc da chay xong)")
        print( "      -> train lai tu dau. Muon giu ket qua cu thi doi ten thu muc run.\n")
    elif drift:
        print("!" * 66)
        print("DUNG LAI: co checkpoint resume duoc, nhung CAU HINH DA DOI.")
        for k, (old, new) in drift.items():
            print(f"   {k:8s} checkpoint = {old!r}   lan nay = {new!r}")
        print()
        print("Ultralytics khi resume se dung lai args cu va BO QUA cai moi,")
        print("nen ban se train tiep cau hinh cu ma khong co dau hieu gi.")
        print()
        print("Chon mot:")
        print(f"  - Train lai tu dau (thuong la dieu ban muon sau khi doi du lieu):")
        print(f"        import shutil; shutil.rmtree(r'{os.path.join(PROJECT, MODEL)}')")
        print(f"  - Hoac giu ket qua cu: doi ten thu muc run do roi chay lai.")
        print("!" * 66)
        raise SystemExit("cau hinh doi -- xoa hoac doi ten thu muc run roi chay lai")
    else:
        print(f"[resume] {last}, tiep tu epoch {ck.get('epoch', 0) + 1}\n")
        model = YOLO(last)
        resume = True

if not resume:
    print(f"[moi] dung {SPEC['cfg']} + nap {SPEC['weights']}\n")
    model = YOLO(SPEC["cfg"], verbose=False)
    model.load(SPEC["weights"])

head = model.model.model[-1]
if getattr(head, "end2end", False):
    head.max_det = MAX_DET
    print(f"[e2e] max_det 300 -> {MAX_DET}\n")

results = model.train(
    data=DATA_YAML, epochs=SPEC["epochs"], imgsz=IMGSZ, batch=SPEC["batch"],
    device=0, workers=WORKERS, seed=SEED, deterministic=False,
    project=PROJECT, name=MODEL, exist_ok=True, resume=resume,
    patience=PATIENCE, amp=True, cache=CACHE, val=True, plots=True,
    freeze=FREEZE if FREEZE else None,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=0.0, translate=0.1, scale=0.5, shear=0.0, perspective=0.0,
    flipud=0.0, fliplr=0.5, mosaic=1.0, mixup=0.0, close_mosaic=10,
)
print("\n[xong] train hoan tat")

# Da cham tran epoch hay da dung som? Voi p2 (chi nap 40% pretrained) day la
# cau hoi quan trong: cham tran nghia la mAP van con dang len khi het epoch,
# tuc con so cuoi cung la mot gioi han nhan tao, khong phai gioi han kien truc.
import pandas as pd
_csv = os.path.join(PROJECT, MODEL, "results.csv")
if os.path.exists(_csv):
    _d = pd.read_csv(_csv)
    _d.columns = [c.strip() for c in _d.columns]
    _n = len(_d)
    _col = next((c for c in _d.columns if "mAP50-95" in c), None)
    if _col:
        _best_ep = int(_d[_col].idxmax()) + 1
        print(f"  epoch da chay  : {_n}/{SPEC['epochs']}")
        print(f"  epoch tot nhat : {_best_ep}  (best.pt la cua epoch nay, khong phai epoch cuoi)")
        if _n < SPEC["epochs"]:
            print(f"\n  -> EARLY STOPPING da kich hoat: {PATIENCE} epoch lien tiep khong cai thien.")
            print(f"     Da bao hoa, con so nay dung la gioi han cua kien truc.")
        elif _best_ep > SPEC["epochs"] - PATIENCE:
            print(f"\n  [luu y] Cham tran {SPEC['epochs']} epoch, va epoch tot nhat nam o cuoi")
            print(f"          -> mAP CON DANG LEN khi het epoch. Con so nay la gioi han cua")
            print(f"          so epoch, KHONG phai cua kien truc. Muon ket luan cong bang thi")
            print(f"          tang EPOCHS_BASE o cell 1 roi chay lai cell nay -- no tu resume.")
        else:
            print(f"\n  -> Cham tran epoch nhung dinh nam o giua -> da bao hoa, ket luan dung.")

## 11. Validate theo giao thức VisDrone

`max_det=500`, không phải 300 mặc định. Ghi ra `summary.json` để cell gom kết
quả đọc được.

In [ ]:
import json, os
from ultralytics import YOLO

best = os.path.join(PROJECT, MODEL, "weights", "best.pt")
assert os.path.exists(best), f"khong thay {best}"

m = YOLO(best)
h = m.model.model[-1]
if getattr(h, "end2end", False):
    h.max_det = MAX_DET

res = m.val(data=DATA_YAML, imgsz=IMGSZ, batch=SPEC["batch"], device=0,
            max_det=MAX_DET, verbose=False)

summary = {
    "model": MODEL,
    "mAP50-95": float(res.box.map),
    "mAP50": float(res.box.map50),
    "mAP75": float(res.box.map75),
    "ap_per_class": {names[i]: float(v) for i, v in enumerate(res.box.maps)},
    "params_M": round(N_PARAMS / 1e6, 3),
    "anchors": N_ANCHORS,
    "end2end_no_nms": END2END,
    "max_det": MAX_DET,
    "weights_transferred": TRANSFER,
    "transfer_frac": round(TRANSFER_FRAC, 4),
    "epochs": SPEC["epochs"], "batch": SPEC["batch"], "imgsz": IMGSZ,
    "seed": SEED, "nc": NC,
    "output_shape_expected": list(OUT_SHAPE_TRAINED),
    "gpu": GPU_NAME,
    "ultralytics": ultralytics.__version__,
}
json.dump(summary, open(os.path.join(PROJECT, MODEL, "summary.json"), "w"),
          indent=2, ensure_ascii=False)

print(f"{MODEL}")
print(f"  mAP50-95 {summary['mAP50-95']:.4f}")
print(f"  mAP50    {summary['mAP50']:.4f}")
print(f"  mAP75    {summary['mAP75']:.4f}")
print("\nAP theo lop:")
for k, v in summary["ap_per_class"].items():
    print(f"  {k:18s} {v:.4f}")

## 12. Export ONNX cho pipeline QCS8550

Đúng thiết lập board yêu cầu, và vá sẵn lỗi đã gặp thật:

- `opset=13` — opset mới sinh op mà QNN đẩy ngược về CPU
- `dynamic=False`, shape tĩnh — bắt buộc để tạo QNN context binary
- **`sanitise_onnx()`** — Ultralytics + onnxslim để tensor output nằm cả trong
  `graph.output` lẫn `value_info`. ONNX Runtime bỏ qua, còn AI Hub **từ chối
  compile**: `Tensors {'output0'} occur in value_info but also in model IO`.
  Lỗi này đã chặn pipeline một lần rồi.
- `nms=False` chỉ truyền cho model cần NMS. Truyền cho model end2end là vô
  nghĩa và có thể làm export lỗi.

In [ ]:
import os, json, hashlib, shutil, warnings
warnings.filterwarnings("ignore")
from ultralytics import YOLO


def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def sanitise_onnx(path):
    """Bo tensor vua nam trong graph IO vua nam trong value_info."""
    import onnx
    mo = onnx.load(path)
    io_names = {t.name for t in mo.graph.input} | {t.name for t in mo.graph.output}
    dupes = [vi.name for vi in mo.graph.value_info if vi.name in io_names]
    if dupes:
        keep = [vi for vi in mo.graph.value_info if vi.name not in io_names]
        del mo.graph.value_info[:]
        mo.graph.value_info.extend(keep)
        onnx.save(mo, path)
    return dupes


OUT = os.path.join(PROJECT, MODEL)
m = YOLO(best)
h = m.model.model[-1]
if getattr(h, "end2end", False):
    h.max_det = MAX_DET

kw = dict(format="onnx", imgsz=IMGSZ, opset=13, dynamic=False,
          simplify=True, batch=1)
if not END2END:
    kw["nms"] = False
p = m.export(**kw)

onnx_path = os.path.join(OUT, f"{MODEL}.onnx")
shutil.move(str(p), onnx_path)
dupes = sanitise_onnx(onnx_path)

import onnxruntime as ort
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
ishape = sess.get_inputs()[0].shape
oshape = sess.get_outputs()[0].shape

meta = {
    "model": MODEL,
    "onnx_sha256_16": sha256(onnx_path)[:16],
    "pt_sha256_16": sha256(best)[:16],
    "input_shape": ishape, "output_shape": oshape,
    "end2end_no_nms": END2END, "max_det": MAX_DET if END2END else None,
    "opset": 13, "imgsz": IMGSZ, "nc": NC,
    "value_info_dupes_removed": len(dupes),
    "pipeline_compatible": not END2END,
}
json.dump(meta, open(os.path.join(OUT, "export.json"), "w"), indent=2)

print(f"[ok] {onnx_path}")
print(f"     input  {ishape}")
print(f"     output {oshape}")
print(f"     sha256 {meta['onnx_sha256_16']}")
if dupes:
    print(f"     da va {len(dupes)} value_info trung (AI Hub se tu choi neu khong va)")

print("\n" + "=" * 64)
if END2END:
    print("KHONG tuong thich truc tiep voi 3-pipeline/detector.py")
    print(f"  output {oshape} = box da decode san [x1,y1,x2,y2,conf,cls]")
    print("  detector.py dang decode dang (1, 4+nc, A) -> cam vao se SAI THAM LANG.")
    print("  Can them nhanh: neu shape[-1]==6 thi bo NMS, chi loc theo conf,")
    print("  roi dua toa do ve he anh goc bang gain/pad nhu cu.")
else:
    print("Tuong thich voi 3-pipeline/detector.py")
    print(f"  output {oshape} - decode nhu cu, NMS chay tren Kryo va do rieng.")
print("=" * 64)

## 13. Gom kết quả từ mọi tab

Chạy cell này ở **bất kỳ tab nào sau khi các tab khác xong**. Nó đọc
`summary.json` của mọi model đã train xong trong cùng thư mục Drive.

Cột `weights nạp` phải có mặt: thiếu nó, bảng này sẽ bị đọc thành "kiến trúc p2
kém hơn", trong khi thực ra p2 chỉ xuất phát sau.

In [ ]:
import json, glob, os
import pandas as pd

rows = []
for sp in sorted(glob.glob(os.path.join(PROJECT, "*", "summary.json"))):
    s = json.load(open(sp))
    ep = os.path.join(os.path.dirname(sp), "export.json")
    e = json.load(open(ep)) if os.path.exists(ep) else {}
    rows.append({
        "model": s["model"],
        "mAP50-95": round(s["mAP50-95"], 4),
        "mAP50": round(s["mAP50"], 4),
        "mAP75": round(s["mAP75"], 4),
        "params(M)": s["params_M"],
        "anchors": s["anchors"] or "e2e",
        "NMS": "khong" if s["end2end_no_nms"] else "can",
        "max_det": s["max_det"],
        "epochs": s["epochs"],
        "batch": s["batch"],
        "weights nap": s["weights_transferred"],
        "onnx output": str(e.get("output_shape", "chua export")),
    })

if not rows:
    print("Chua co model nao xong.")
else:
    df = pd.DataFrame(rows).sort_values("mAP50-95", ascending=False)
    display(df)
    out_csv = os.path.join(PROJECT, "comparison.csv")
    df.to_csv(out_csv, index=False)
    print(f"\n[ok] {out_csv}")
    print(f"[ok] {len(rows)}/4 model da xong: {', '.join(r['model'] for r in rows)}")

    print("\nDoc bang nay can nho:")
    print("  - Cot 'weights nap': model nao thap hon ~75% la xuat phat sau,")
    print("    khong phai kien truc kem hon.")
    print("  - Cot 'onnx output': dang (1,N,6) la box decode san, KHONG chay NMS;")
    print("    dang (1,4+nc,A) moi cam thang vao pipeline hien tai duoc.")

## 14. Chẩn đoán: 4 model có **thực sự so sánh được** với nhau không

Bảng mAP ở cell 13 chỉ có nghĩa nếu bốn model được huấn luyện tương đương.
Chúng chạy ở bốn tab khác nhau, batch tự động khác nhau, và có thể chạy ở
những phiên bản notebook khác nhau — nên điều đó **không hiển nhiên**.

Cột quyết định là **`buoc_gradient`** = số iteration × số epoch. Đó là số lần
model thực sự được cập nhật. Hai model lệch nhau vài lần ở cột này thì bảng
mAP đang so "model nào được train nhiều hơn", không phải "kiến trúc nào tốt
hơn".

In [ ]:
import os, glob, json, yaml, math
import pandas as pd

rows = []
for run in sorted(glob.glob(os.path.join(PROJECT, "*"))):
    ap = os.path.join(run, "args.yaml")
    rp = os.path.join(run, "results.csv")
    if not (os.path.exists(ap) and os.path.exists(rp)):
        continue
    a = yaml.safe_load(open(ap))
    d = pd.read_csv(rp)
    d.columns = [c.strip() for c in d.columns]

    n_train = None
    try:
        dy = yaml.safe_load(open(a["data"]))
        tr = dy["train"]
        if os.path.isdir(tr):
            n_train = len([f for f in os.listdir(tr)
                           if f.lower().endswith((".jpg", ".jpeg", ".png"))])
    except Exception:
        pass

    ep_run = len(d)
    bs = a.get("batch")
    it_per_ep = math.ceil(n_train / bs) if (n_train and bs) else None
    steps = it_per_ep * ep_run if it_per_ep else None

    tcol = next((c for c in d.columns if c == "time"), None)
    total_min = (d[tcol].iloc[-1] / 60) if tcol else None

    mcol = next((c for c in d.columns if "mAP50-95" in c), None)
    best_ep = int(d[mcol].idxmax()) + 1 if mcol else None

    rows.append({
        "model": os.path.basename(run),
        "batch": bs,
        "epoch_dat": a.get("epochs"),
        "epoch_chay": ep_run,
        "iter/epoch": it_per_ep,
        "buoc_gradient": steps,
        "phut": round(total_min, 1) if total_min else None,
        "s/epoch": round(total_min * 60 / ep_run, 1) if total_min else None,
        "epoch_tot_nhat": best_ep,
        "mAP50-95": round(float(d[mcol].max()), 4) if mcol else None,
        "cache": a.get("cache"),
        "imgsz": a.get("imgsz"),
    })

if not rows:
    print(f"Khong thay run nao trong {PROJECT}")
else:
    df = pd.DataFrame(rows)
    display(df)

    print("\n" + "=" * 66)
    st = [r["buoc_gradient"] for r in rows if r["buoc_gradient"]]
    if len(st) > 1 and max(st) / min(st) > 2:
        lo = min(rows, key=lambda r: r["buoc_gradient"] or 1e9)
        hi = max(rows, key=lambda r: r["buoc_gradient"] or 0)
        print(f"BANG mAP O CELL 13 CHUA SO SANH DUOC")
        print(f"  {hi['model']} duoc cap nhat {hi['buoc_gradient']} lan")
        print(f"  {lo['model']} chi duoc {lo['buoc_gradient']} lan")
        print(f"  -> chenh {max(st)/min(st):.0f}x. Su khac biet mAP giua hai model nay")
        print(f"     phan anh so buoc train, khong phai kien truc.")
        print(f"  Cach sua: dat cung batch cho ca 4 model roi train lai,")
        print(f"     hoac chinh epochs de buoc_gradient xap xi bang nhau.")
    else:
        print("So buoc gradient tuong duong -> bang mAP so sanh duoc.")

    if any(r["imgsz"] != rows[0]["imgsz"] for r in rows):
        print("\n!! imgsz khac nhau giua cac run -> khong so duoc.")
    if len({r["epoch_dat"] for r in rows}) > 2:
        print("\n!! epochs dat khac nhau nhieu hon du kien (p2 = 1.5x la binh thuong).")
    print("=" * 66)

## 15. Chẩn đoán khi mAP đứng yên

Chạy cell này khi thấy loss giảm đều mà mAP val không nhúc nhích. Nó phân biệt
ba nguyên nhân khác hẳn nhau — và cách sửa của ba cái này ngược nhau, nên đoán
sai là sửa sai.

| Dấu hiệu | Nguyên nhân | Cách sửa |
|---|---|---|
| mAP trên **train** cao, trên val thấp | overfit vì ít cảnh | giảm epoch, tăng augment, thêm sequence |
| Phần lớn box **< 8 px** ở 640 | vật thể quá nhỏ | tăng `imgsz`, giảm `mosaic`, dùng p2 |
| Ảnh vẽ nhãn ra **lệch chỗ** | lỗi chuyển đổi nhãn | sửa converter, train lại |

In [ ]:
import os, glob, random, math
import numpy as np, cv2, yaml
from ultralytics import YOLO

run = os.path.join(PROJECT, MODEL)
best = os.path.join(run, "weights", "best.pt")
dy = yaml.safe_load(open(DATA_YAML))

# ---- 1. Bao nhieu CANH khac nhau, khong phai bao nhieu anh ---------------
# Ten file la <ten_sequence>_<so_frame>. Anh cach nhau 3 frame gan nhu trung
# nhau, nen so anh khong phai thuoc do da dang -- so sequence moi la.
import re
# Ten file la <sequence>_<7 chu so frame>[_t<2 chu so o>]. Ten sequence cua
# VisDrone tu no da chua dau gach duoi (uav0000013_00000_v) nen khong the tach
# bang split("_") -- phai cat dung phan duoi bang regex.
_SEQ_RE = re.compile(r"_\d{7}(?:_t\d{2})?$")


def seq_of(stem):
    return _SEQ_RE.sub("", stem)


def images_of(entry):
    """dy[split] co the la thu muc hoac file .txt liet ke duong dan."""
    if os.path.isdir(entry):
        return [os.path.join(entry, f) for f in os.listdir(entry)
                if f.endswith(IMG_EXT)]
    return [l.strip() for l in open(entry) if l.strip()]


seq_by_split = {}
for split in ("train", "val"):
    paths = images_of(dy[split])
    stems = [os.path.splitext(os.path.basename(p))[0] for p in paths]
    seqs = {seq_of(s) for s in stems}
    seq_by_split[split] = seqs
    print(f"  {split:5s} {len(stems):6d} anh tu {len(seqs):3d} sequence "
          f"({len(stems)/max(len(seqs),1):.0f} anh/sequence)")
both = seq_by_split["train"] & seq_by_split["val"]
print(f"  sequence xuat hien o CA HAI: {len(both)}"
      f"  (khac 0 la ro ri train/val -> mAP val khong tin duoc)")

# ---- 2. Vat the con lai bao nhieu pixel o 640 ---------------------------
train_paths = images_of(dy["train"])
sizes = []
for src in random.Random(0).sample(train_paths, min(300, len(train_paths))):
    lf = os.path.join(os.path.dirname(os.path.dirname(src)), "labels",
                      os.path.splitext(os.path.basename(src))[0] + ".txt")
    if not os.path.exists(lf):
        continue
    im = cv2.imread(src)
    if im is None:
        continue
    H, W = im.shape[:2]
    gain = IMGSZ / max(H, W)          # letterbox giu ti le
    for line in open(lf):
        p = line.split()
        if len(p) < 5:
            continue
        sizes.append(max(float(p[3]) * W, float(p[4]) * H) * gain)

sizes = np.array(sizes)
print(f"\n  Canh dai cua vat the, do bang pixel TRONG anh 640 ({len(sizes)} box):")
for q in (50, 75, 90, 99):
    print(f"    p{q:<3d} {np.percentile(sizes, q):6.1f} px")
tiny = (sizes < 8).mean() * 100
print(f"    duoi  8 px: {tiny:5.1f} %   <- stride-8 (P3) khong bat duoc")
print(f"    duoi  4 px: {(sizes < 4).mean()*100:5.1f} %   <- stride-4 (P2) cung khong")
print(f"    Mosaic ghep 4 anh nen con co them ~2x nua trong luc train.")

# ---- 3. mAP tren TRAIN so voi tren val ---------------------------------
# Chenh lech lon = overfit. Bang nhau ma cung thap = du lieu/nhan co van de,
# khong phai model.
if os.path.exists(best):
    m = YOLO(best)
    h = m.model.model[-1]
    if getattr(h, "end2end", False):
        h.max_det = MAX_DET
    tmp = "/content/_diag.yaml"
    yaml.safe_dump({**dy, "val": dy["train"]}, open(tmp, "w"))
    r_tr = m.val(data=tmp, imgsz=IMGSZ, batch=SPEC["batch"], device=0,
                 max_det=MAX_DET, verbose=False)
    r_va = m.val(data=DATA_YAML, imgsz=IMGSZ, batch=SPEC["batch"], device=0,
                 max_det=MAX_DET, verbose=False)
    print(f"\n  mAP50-95 tren train {r_tr.box.map:.4f}")
    print(f"  mAP50-95 tren val   {r_va.box.map:.4f}")
    gap = r_tr.box.map / max(r_va.box.map, 1e-9)
    print(f"  ty le {gap:.2f}x")
    print()
    if gap > 2.0:
        print("  -> OVERFIT. Model thuoc canh train. Them sequence, giam epoch,")
        print("     hoac tang augment. Train lau hon se khong giup gi.")
    elif r_va.box.map < 0.10:
        print("  -> KHONG phai overfit: train cung thap. Van de nam o du lieu")
        print("     hoac o nhan, khong phai o so epoch. Xem anh o buoc 4.")
    else:
        print("  -> Chenh lech binh thuong.")

# ---- 4. Ve nhan len anh: cach chac chan nhat de thay loi chuyen doi -----
os.makedirs("/content/diag", exist_ok=True)
picked = random.Random(1).sample(train_paths, 4)
tiles = []
for src in picked:
    f = os.path.basename(src)
    im = cv2.imread(src)
    H, W = im.shape[:2]
    lf = os.path.join(os.path.dirname(os.path.dirname(src)), "labels",
                      os.path.splitext(f)[0] + ".txt")
    n = 0
    for line in open(lf):
        p = line.split()
        if len(p) < 5:
            continue
        c, cx, cy, bw, bh = int(p[0]), *map(float, p[1:5])
        x1, y1 = int((cx - bw / 2) * W), int((cy - bh / 2) * H)
        x2, y2 = int((cx + bw / 2) * W), int((cy + bh / 2) * H)
        cv2.rectangle(im, (x1, y1), (x2, y2), (0, 255, 0), 2)
        n += 1
    cv2.putText(im, f"{f}  {n} box", (10, 30), cv2.FONT_HERSHEY_SIMPLEX,
                0.9, (0, 255, 255), 2)
    tiles.append(cv2.resize(im, (960, 540)))
grid = np.vstack([np.hstack(tiles[:2]), np.hstack(tiles[2:])])
out = "/content/diag/labels_on_images.jpg"
cv2.imwrite(out, grid)
print(f"\n  [xem anh nay] {out}")
print("  Box phai om dung xe/nguoi. Lech he thong = loi converter.")
from IPython.display import Image, display as _d
_d(Image(filename=out, width=900))

---

## Sau khi cả 4 model xong

**Bắt buộc trước khi dùng v26 trong pipeline.** `3-pipeline/detector.py` hiện
chỉ decode `(1, 4+nc, A)`. Hai model v26 trả `(1, N, 6)` đã decode sẵn. Cắm vào
mà không sửa thì **không có lỗi nào được báo** — chỉ là kết quả sai. Cần thêm
nhánh: `shape[-1] == 6` → tách `[x1, y1, x2, y2, conf, cls]`, bỏ NMS, lọc theo
`conf`, rồi đưa toạ độ về hệ ảnh gốc bằng `gain`/`pad` như cũ.

**Việc đáng làm nhất sau đó.** Bỏ NMS là bỏ **18 ms/frame** trên CPU — khoản
cắt lớn nhất còn lại trong frame budget (inference chỉ 47 ms). Nhưng phải kiểm
tra thật: đầu end-to-end dùng `topk`, và `topk` có thể bị QNN đẩy về CPU. Chạy
compile job trên AI Hub rồi đọc `n_ops_fallback` **trước khi** tin vào con số
này — nếu > 0 thì phần tiết kiệm sẽ bị trả lại ở chỗ khác.

**Mỗi dòng kết quả phải ghi kèm** `max_det` (500, không phải 300), tỉ lệ trọng
số nạp được, `imgsz`, `seed`, và sha256 của ONNX. Thiếu bất kỳ cái nào là hai
bảng không so được với nhau.